課題1では，シンプルなニューラルネットワークに対して，勾配法（最急降下法）で結合強度を学習しました．

ですが，プログラムを実行して結果が出るまでに時間がかかったと思います．

結合強度の学習には，逆誤差伝播法を用いたほうが高速に学習できます．実際に実装して確認してみましょう．

**ノートブックの最後のほうに，自分でプログラムを書いてほしいところがありますが，課題1ができていればそれほど難しくはありません．**

授業資料と`sc-fp-and-bp.ipynb`で確認したように，逆誤差伝播法でも1つの計算につき，順伝播と逆伝播を用意することで，簡単に実装できます．

逆誤差伝播法を実装するのに必要なレイヤを準備していきます．

# 活性化関数レイヤの実装

このノートブックでは，活性化関数として，ReLU(Rectified Linear Unit)関数を用います．

活性化関数もレイヤとして実装していきましょう．

`sc-fp-and-bp.ipynb`の乗算レイヤ，加算レイヤと同じく，レイヤを表すクラス内に，初期化，順伝播，逆伝播の3つの関数を用意します．

## ReLUレイヤ

ReLUは以下の式で表せます．

$$
    y =
        \begin{cases}
            x & (x > 0) \\
            0 & (x \leq 0) \\
        \end{cases}
$$

この式に対して， $x$に関する$y$の微分は以下のように求められます．

$$
    \frac{\partial y}{\partial x} =
        \begin{cases}
            1 & (x > 0) \\
            0 & (x \leq 0) \\
        \end{cases}
$$

これは，つまり，入力$x$が0より大きければ，逆伝播のときには上流の値をそのまま下流へ伝えることになります．

逆に，入力$x$が0以下のときは0を返すので，逆伝播のときに上流の値が下流へ伝わらないことになります（上流からの値に0を掛けることになるので）．

それでは，ReLUレイヤを実装してみます．

In [1]:
import numpy as np

# ReLUレイヤのクラス
class Relu:
    # 初期化関数
    def __init__(self):
        self.mask = None

    # 順伝播の関数
    def forward(self, x):
        self.mask = (x <= 0) # 入力配列xの中身が0以下かどうかを調べてmaskに記録
        out = x.copy() # 戻り値となる配列outに入力配列xをコピーする
        out[self.mask] = 0 # xの中身が0以下だったところ（self.mask=Trueのところ）を0に書き換える

        return out

    # 逆伝播の関数
    def backward(self, dout): # doutは上流から伝わってきた微分値の配列
        dout[self.mask] = 0 # xの中身が0以下だったところを0に書き換える
        dx = dout

        return dx

上のプログラムに出てくる`mask`というのは，`True`/`False`からなるNumPy配列です．プログラム中では，配列の中身が0以下かどうか，を表すのに用いています．

ReLU関数は入力が0以下のときには0を返すので，`mask`に入力配列`x`の要素が0以下かどうかをブーリアンで記録しておいて，入力配列の内容がコピーされた戻り値の配列`out`を一気に書き換えるという処理をしています．

いまいちわからない，という人は，以下のプログラムを実行してみるといいかもしれません．

In [2]:
import numpy as np

x = np.array([[1.0, -0.5], [-2.0, 3.0]])
print(x)

mask = (x <= 0) # ここで，xの中身が0以下かを見て，0以下ならTrueを，そうでなければFalseをmaskに入れる
print(mask) # xの中身で0以下のところにTrue，そうでないところにFalseが入っているはず

x[mask] = 0 # 0以下の要素を0に書き換え
print(x)

[[ 1.  -0.5]
 [-2.   3. ]]
[[False  True]
 [ True False]]
[[1. 0.]
 [0. 3.]]


活性化関数は他にも，課題1で登場した，シグモイド関数や恒等関数もありますが，レイヤとしての実装例は省略します．

# Affine/Softmaxレイヤの実装

次に，ニューラルネットワークの計算として行われる内積の計算と，ニューラルネットワークの最終的な出力に必要なソフトマックス関数のレイヤを実装します．

## Affineレイヤ

Affineというのは，行列の内積を表す「アフィン変換」のアフィンのことです．もしかしたら，画像処理系の授業で聞いたことがあったり，実装したことがあるかも？

では，アフィン変換を行うAffineレイヤを実装します．

ニューラルネットワークの各ノードでは，以下のような行列計算が行われていました．

$$
    \boldsymbol{Y} = \boldsymbol{X} \boldsymbol{W} + \boldsymbol{B}
$$

※変数名が課題1のノートブックと一部違いますが，置き換えて考えてください．

これも計算グラフで表すと，順伝播と逆伝播をどのように実装すればよいのかがわかります．

ですが，計算グラフの解説を入れてしまうと，授業時間内で逆誤差伝播法の実装部分にたどり着けなくなってしまうので，今回は省略します．

In [3]:
import numpy as np

# Affineレイヤのクラス
class Affine:
    # 初期化関数
    def __init__(self, W, b):
        self.W =W
        self.b = b
        
        self.x = None
        self.original_x_shape = None
        # 重み・バイアスパラメータの微分
        self.dW = None
        self.db = None

    # 順伝播の関数
    def forward(self, x):
        # テンソル対応
        self.original_x_shape = x.shape
        x = x.reshape(x.shape[0], -1)
        self.x = x

        out = np.dot(self.x, self.W) + self.b # ここが，内積計算の部分

        return out

    # 逆伝播の関数
    def backward(self, dout):
        dx = np.dot(dout, self.W.T)
        self.dW = np.dot(self.x.T, dout)
        self.db = np.sum(dout, axis=0)
        
        dx = dx.reshape(*self.original_x_shape)  # 入力データの形状に戻す（テンソル対応）
        return dx

## Softmax-with-loss レイヤ

ニューラルネットワークの出力層となるソフトマックス関数は以下の式で表します．

$$
    y_k = \frac{exp(a_k)}{\sum_{i=1}^{n}exp(a_i)}
$$

ソフトマックス関数は，入力された値を正規化して出力する関数です．ここでの正規化というのは，**出力の和が1になるように変形させる**ことを言います．

たとえば，前回のアヤメ(iris)データのように3つに分類する場合は，出力は3つあり，その3つを足し算すると1になるようにするということです．

Softmaxレイヤへの入力も，出力と同じく，分類したいクラス数となります．アヤメデータならば，Softmaxレイヤの入力は3つ，出力も3つです．

ここから，Softmaxレイヤの実装をしますが，出力層なので損失関数のことも考えなければいけません．

ここでは，損失関数として交差エントロピー誤差を計算することにし，損失関数も含めたレイヤということで，Softmax-with-lossレイヤを実装します．

なお，ソフトマックス関数と交差エントロピー誤差の計算グラフは，大変複雑ですので導出の仕方は省略します．

In [4]:
import numpy as np

# ソフトマックス関数と損失関数のクラス
class SoftmaxWithLoss:
    def __init__(self):
        self.loss = None
        self.y = None # softmaxの出力
        self.t = None # 教師データ

    # 順伝播
    def forward(self, x, t):
        self.t = t # 教師信号の出力
        self.y = softmax(x) # ネットワークの出力
        self.loss = cross_entropy_error(self.y, self.t) # 交差エントロピー誤差を計算
        
        return self.loss

    # 逆伝播
    def backward(self, dout=1):
        batch_size = self.t.shape[0]
        if self.t.size == self.y.size: # 教師データがone-hot-vectorの場合
            dx = (self.y - self.t) / batch_size
        else:
            dx = self.y.copy()
            dx[np.arange(batch_size), self.t] -= 1
            dx = dx / batch_size
        
        return dx
    
    
# 活性化関数（ソフトマックス関数）
# 課題1で用いたのと同じ定義
def softmax(x):
    if x.ndim == 2:
        x = x.T
        x = x - np.max(x, axis=0)
        y = np.exp(x) / np.sum(np.exp(x), axis=0)
        return y.T 

    x = x - np.max(x) # オーバーフロー対策
    return np.exp(x) / np.sum(np.exp(x))

# 交差エントロピー誤差（バッチ学習対応版）
# 課題1で用いたのと同じ定義
def cross_entropy_error(y, t):
    if y.ndim == 1:
        t = t.reshape(1, t.size)
        y = y.reshape(1, y.size)
        
    # 教師データがone-hot-vectorの場合、正解ラベルのインデックスに変換
    if t.size == y.size:
        t = t.argmax(axis=1)
             
    batch_size = y.shape[0]
    return -np.sum(np.log(y[np.arange(batch_size), t] + 1e-7)) / batch_size

# 逆誤差伝播法の実装

ここまで実装したレイヤを組み合わせて，逆誤差伝播法（を用いるニューラルネットワーク）を実装できます．

まずは，必要なクラスと関数を定義します．ほとんどはこれまでに登場したものです．

In [5]:
import numpy as np

# 活性化関数ReLUレイヤのクラス
class Relu:
    # 初期化関数
    def __init__(self):
        self.mask = None

    # 順伝播の関数
    def forward(self, x):
        self.mask = (x <= 0) # 入力配列xの中身が0以下かどうかを調べてmaskに記録
        out = x.copy() # 戻り値となる配列outに入力配列xをコピーする
        out[self.mask] = 0 # xの中身が0以下だったところ（self.mask=Trueのところ）を0に書き換える

        return out

    # 逆伝播の関数
    def backward(self, dout): # doutは上流から伝わってきた微分値の配列
        dout[self.mask] = 0 # xの中身が0以下だったところを0に書き換える
        dx = dout

        return dx
    
    
# 内積を計算するAffineレイヤのクラス
class Affine:
    # 初期化関数
    def __init__(self, W, b):
        self.W =W
        self.b = b
        
        self.x = None
        self.original_x_shape = None
        # 重み・バイアスパラメータの微分
        self.dW = None
        self.db = None

    # 順伝播の関数
    def forward(self, x):
        # テンソル対応
        self.original_x_shape = x.shape
        x = x.reshape(x.shape[0], -1)
        self.x = x

        out = np.dot(self.x, self.W) + self.b # ここが，ネットワークの内積計算の部分

        return out

    # 逆伝播の関数
    def backward(self, dout):
        dx = np.dot(dout, self.W.T)
        self.dW = np.dot(self.x.T, dout)
        self.db = np.sum(dout, axis=0)
        
        dx = dx.reshape(*self.original_x_shape)  # 入力データの形状に戻す（テンソル対応）
        return dx

    
# 出力層でソフトマックス関数(Softmax)と交差エントロピー誤差(Loss)のレイヤのクラス
class SoftmaxWithLoss:
    def __init__(self):
        self.loss = None
        self.y = None # softmaxの出力
        self.t = None # 教師データ

    def forward(self, x, t):
        self.t = t
        self.y = softmax(x)
        self.loss = cross_entropy_error(self.y, self.t) # 交差エントロピー誤差
        
        return self.loss

    def backward(self, dout=1):
        batch_size = self.t.shape[0]
        if self.t.size == self.y.size: # 教師データがone-hot-vectorの場合
            dx = (self.y - self.t) / batch_size
        else:
            dx = self.y.copy()
            dx[np.arange(batch_size), self.t] -= 1
            dx = dx / batch_size
        
        return dx
    
## ここからは関数の定義    

# ソフトマックス関数
def softmax(x):
    if x.ndim == 2:
        x = x.T
        x = x - np.max(x, axis=0)
        y = np.exp(x) / np.sum(np.exp(x), axis=0)
        return y.T 

    x = x - np.max(x) # オーバーフロー対策
    return np.exp(x) / np.sum(np.exp(x))

# 交差エントロピー誤差
def cross_entropy_error(y, t):
    if y.ndim == 1:
        t = t.reshape(1, t.size)
        y = y.reshape(1, y.size)
        
    # 教師データがone-hot-vectorの場合、正解ラベルのインデックスに変換
    if t.size == y.size:
        t = t.argmax(axis=1)
     
    batch_size = y.shape[0]
    return -np.sum(np.log(y[np.arange(batch_size), t] + 1e-7)) / batch_size

# 数値微分で勾配を計算する関数
def numerical_gradient(f, x):
    h = 1e-4 # 0.0001
    grad = np.zeros_like(x)
    
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        idx = it.multi_index
        tmp_val = x[idx]
        x[idx] = float(tmp_val) + h
        fxh1 = f(x) # f(x+h)
        
        x[idx] = tmp_val - h 
        fxh2 = f(x) # f(x-h)
        grad[idx] = (fxh1 - fxh2) / (2*h)
        
        x[idx] = tmp_val # 値を元に戻す
        it.iternext()   
        
    return grad

そして，2層ニューラルネットワークのクラスを用意します．

逆誤差伝播法を使って結合強度を学習するので，勾配の求め方が課題1とは違い，逆伝播を使った書き方になっています．

（このあと，数値微分との比較を行うので，数値微分の関数も含めています）

In [6]:
import numpy as np
from collections import OrderedDict

# 2層ニューラルネットワークのクラス
class TwoLayerNet:

    # 初期化関数
    def __init__(self, input_size, hidden_size, output_size, weight_init_std = 0.01):
        # 重みの初期化
        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(input_size, hidden_size)
        self.params['b1'] = np.zeros(hidden_size)
        self.params['W2'] = weight_init_std * np.random.randn(hidden_size, output_size) 
        self.params['b2'] = np.zeros(output_size)

        # レイヤの生成
        self.layers = OrderedDict() # 順番付きディクショナリ，という仕組みを使ってレイヤを生成する
        # self.layersに追加した順序でレイヤを呼び出すことができる

        self.layers['Affine1'] = Affine(self.params['W1'], self.params['b1']) # 1層目の内積計算レイヤ
        self.layers['Relu1'] = Relu() # 1層目の活性化関数レイヤ
        self.layers['Affine2'] = Affine(self.params['W2'], self.params['b2']) # 2層目の内積計算レイヤ
        self.lastLayer = SoftmaxWithLoss() # 出力層のレイヤ
        
    # ネットワークの計算をする関数
    def predict(self, x):
        for layer in self.layers.values():
            x = layer.forward(x) # ここで，順番付きディクショナリの機能により，self.layers内のレイヤを宣言した順に呼び出している．
            # 順番付きディクショナリを使うことで，レイヤがいくつになってもここを書き換える必要がなくて楽になる
        
        return x
        
    # 損失関数を計算する関数
    # x:入力データ, t:教師データ
    def loss(self, x, t):
        y = self.predict(x)
        return self.lastLayer.forward(y, t)
    
    # 認識精度を計算する関数
    def accuracy(self, x, t):
        y = self.predict(x)
        y = np.argmax(y, axis=1)
        if t.ndim != 1 : t = np.argmax(t, axis=1)
        
        accuracy = np.sum(y == t) / float(x.shape[0])
        return accuracy
        
    # 勾配を計算（数値微分バージョン）
    # x:入力データ, t:教師データ
    def numerical_gradient(self, x, t):
        loss_W = lambda W: self.loss(x, t)
        
        grads = {}
        grads['W1'] = numerical_gradient(loss_W, self.params['W1'])
        grads['b1'] = numerical_gradient(loss_W, self.params['b1'])
        grads['W2'] = numerical_gradient(loss_W, self.params['W2'])
        grads['b2'] = numerical_gradient(loss_W, self.params['b2'])
        
        return grads
        
    # 勾配を計算（誤差逆伝播法バージョン）
    def gradient(self, x, t):
        # forward
        self.loss(x, t)

        # backward
        dout = 1
        dout = self.lastLayer.backward(dout) # 最後に追加したレイヤ，つまり，SoftmaxWithLossレイヤの逆伝播を計算
        
        layers = list(self.layers.values())
        layers.reverse() # 順番付きディクショナリの順序を逆にする
        for layer in layers: # 順序が逆になったので，ネットワークの後ろ（ネットワーク図の右側）から逆伝播の計算をしていく
            dout = layer.backward(dout) # 逆伝播の計算自体はTwoLayerNetクラスで実装済みなので，ここで計算のためのプログラムを書かなくてよい

        # 設定
        grads = {}
        grads['W1'], grads['b1'] = self.layers['Affine1'].dW, self.layers['Affine1'].db
        grads['W2'], grads['b2'] = self.layers['Affine2'].dW, self.layers['Affine2'].db

        return grads

もしかしたら，「こんなに簡単でいいの？」という感じがするかもしれません．

各レイヤのクラスにおいて，逆伝播の関数`backward()`で微分を計算できるようにしてあるので，微分した値を次のレイヤへ渡すだけで計算ができてしまうのです．

課題1のノートブックと見比べると，predict関数やgradient関数がレイヤを用いてシンプルに書けていることがわかります．

いま実装したものは2層のネットワークですが，層を増やしたい場合には，単純に`__init__`の中の「レイヤの生成」部分で必要なだけレイヤを用意すればよいのです．

## 逆誤差伝播法の勾配を確認する

実装した逆誤差伝播法で本当に勾配が計算できているのか，前回用いた数値微分による方法と比較することで，確認してみます．

今回も，実際のデータとして，アヤメ(iris)データを使います．

このノートブックと同じディレクトリに，`iris.csv`，`iris_train.csv`，`iris_test.csv`があることを確認してから，以下のセルを実行してください．

In [7]:
import pandas as pd

# irisデータの読み込み
iris = pd.read_csv('./iris.csv', header=None)
iris.columns = ["Sepal Length（がく片の長さ）", "Sepal Width（がく片の幅）", "Petal Length（花びらの長さ）", "Petal Width（花びらの幅）", "品種名"]

# iris データを表示する
iris

,Sepal Length（がく片の長さ）,Sepal Width（がく片の幅）,Petal Length（花びらの長さ）,Petal Width（花びらの幅）,品種名
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,Iris-virginica
146,6.3,2.5,5.0,1.9,Iris-virginica
147,6.5,3.0,5.2,2.0,Iris-virginica
148,6.2,3.4,5.4,2.3,Iris-virginica


では，数値微分と逆誤差伝播法の結果を比較してみます．

In [8]:
import pandas as pd
import numpy as np

# データの読み込み
# 学習データ
train = pd.read_csv('./iris_train.csv', header=None)
train_np = np.array(train.values.flatten())
train_array = np.reshape(train_np, (train.shape[0], train.shape[1]))
x_train = train_array[0:, 0:4].astype(np.float32)
t_train = train_array[0:, 4:].astype(np.float32)

# 教師データ
test = pd.read_csv('./iris_test.csv', header=None)
test_np = np.array(test.values.flatten())
test_array = np.reshape(test_np, (test.shape[0], test.shape[1]))
x_test = test_array[0:, 0:4].astype(np.float32)
t_test = test_array[0:, 4:].astype(np.float32)

# 2層ニューラルネットワークをつくる
# 入力層のニューロンは4，隠れ層のニューロンは10，出力層のニューロンは3
network = TwoLayerNet(input_size=4, hidden_size=10, output_size=3)

# 学習用データを5つずつ読み込む
x_batch = x_train[:5]
t_batch = t_train[:5]

# 数値微分による勾配計算
grad_numerical = network.numerical_gradient(x_batch, t_batch)
# 逆誤差伝播法による勾配計算
grad_backprop = network.gradient(x_batch, t_batch)

# 数値微分，逆誤差伝播法で求めた勾配の差を表示する
# 表示された差が小さいと，逆誤差伝播法が適切に実装できたということになる
print("数値微分と逆誤差伝播法で求めた勾配の差を確認")
for key in grad_numerical.keys():
    diff = np.average( np.abs(grad_backprop[key] - grad_numerical[key]) )
    print(key + ":" + str(diff))

数値微分と逆誤差伝播法で求めた勾配の差を確認
W1:2.6558550371413442e-09
b1:1.051736997508988e-09
W2:3.234392066718897e-09
b2:1.3347600054854544e-07


上のプログラムを実装した結果，表示された数値が小さければ，数値微分で求めた微分値との差が小さいということになり，逆誤差伝播法でも勾配をきちんと計算できていると言えます．

## 課題(1) 逆誤差伝播法で結合荷重を学習する

それでは，いよいよ，逆誤差伝播法を用いたニューラルネットワークで，アヤメデータの分類に挑戦します．

必要なクラスと関数の定義は，上記で行っているのでここでは省略します．

一部，自分でプログラムを書いてほしいところがあります．課題1`sc2025-neural-network.ipynb`の一番最後のセルを参考にするとよいです．

In [9]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# データの読み込み
# 学習データ
train = pd.read_csv('./iris_train.csv', header=None)
train_np = np.array(train.values.flatten())
train_array = np.reshape(train_np, (train.shape[0], train.shape[1]))
x_train = train_array[0:, 0:4].astype(np.float32)
t_train = train_array[0:, 4:].astype(np.float32)

# 教師データ
test = pd.read_csv('./iris_test.csv', header=None)
test_np = np.array(test.values.flatten())
test_array = np.reshape(test_np, (test.shape[0], test.shape[1]))
x_test = test_array[0:, 0:4].astype(np.float32)
t_test = test_array[0:, 4:].astype(np.float32)

# 2層ニューラルネットワークをつくる
network = TwoLayerNet(input_size=4, hidden_size=10, output_size=3)

iters_num = 1000 # 繰り返しの回数
train_size = x_train.shape[0]
batch_size = 5 # データを5個ずつ読む
learning_rate = 0.1 # 学習率

train_loss_list = []
train_acc_list = []
test_acc_list = []

iter_per_epoch = max(train_size / batch_size, 1)

# 繰り返しの回数分だけ，学習を行う
for i in range(iters_num):
    batch_mask = np.random.choice(train_size, batch_size)
    x_batch = x_train[batch_mask]
    t_batch = t_train[batch_mask]
    
    # 勾配の計算：逆誤差伝播法で行う．numerical_gradientは使わない．
    grad = "ここを自分で書く"
    
    # 結合荷重の更新
    for key in ('W1', 'b1', 'W2', 'b2'):
        network.params[key] -= "ここを自分で書く"
    
    # 損失関数で交差エントロピー誤差を求める
    loss = "ここを自分で書く"
    train_loss_list.append(loss)
    
    # データ1週ごとに認識精度を計算する
    if i % iter_per_epoch == 0:
        train_acc = network.accuracy(x_train, t_train)
        test_acc = network.accuracy(x_test, t_test)
        train_acc_list.append(train_acc)
        test_acc_list.append(test_acc)
        print("train acc, test acc | " + str(train_acc) + ", " + str(test_acc)) # 精度の表示
        
# グラフの描画
markers = {'train': 'o', 'test': 's'}
x = np.arange(len(train_acc_list))
plt.plot(x, train_acc_list, label='train acc')
plt.plot(x, test_acc_list, label='test acc', linestyle='--')
plt.xlabel("epochs")
plt.ylabel("accuracy")
plt.ylim(0, 1.0)
plt.legend(loc='lower right')
# plt.savefig('bp-iris.png')
plt.show()

UFuncTypeError: ufunc 'subtract' did not contain a loop with signature matching types (dtype('float64'), dtype('<U8')) -> None

`sc2025-neural-network.ipynb`と同様にアヤメ(iris)データを使って学習ができました．しかも，実行終了までの時間が短くなったことがわかったと思います．

ただし，`sc2025-neural-network.ipynb`で実行したときと比べると，エポックごとの精度のばらつきが大きくなっているかもしれません（実行のタイミングによっては，ばらつきが小さいこともあります）

### 課題(2) パラメータを変えてみる

パラメータを変えてみて，実行結果にどのような変化が現れるかを確認してください．以下のパラメータを変更することができます．

**第1層（隠れ層）の数を変更してみる**

> network = TwoLayerNet(input_size=4, hidden_size=10, output_size=3)

hidden_sizeの値を変えて実行する（input_sizeとoutput_sizeは変えないこと）

**繰り返しの回数を変更してみる**

> iters_num = 1000  # 繰り返しの回数

**バッチ学習の，読み込むデータの個数を変えてみる**

> batch_size = 5 # データを5個ずつ読む

バッチ学習で読み込むデータの個数はエポック数に影響することに注意．

**学習率の値を変えてみる**

> learning_rate = 0.1 # 学習率

以下のセルは上記の **課題(1)** をコピーしたものです．

以下のセル内のパラメータを変更して，実行してください．パラメータを変更した箇所には，コメントを入れておいてください（採点するために必要です）

In [ ]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# データの読み込み
# 学習データ
train = pd.read_csv('./iris_train.csv', header=None)
train_np = np.array(train.values.flatten())
train_array = np.reshape(train_np, (train.shape[0], train.shape[1]))
x_train = train_array[0:, 0:4].astype(np.float32)
t_train = train_array[0:, 4:].astype(np.float32)

# 教師データ
test = pd.read_csv('./iris_test.csv', header=None)
test_np = np.array(test.values.flatten())
test_array = np.reshape(test_np, (test.shape[0], test.shape[1]))
x_test = test_array[0:, 0:4].astype(np.float32)
t_test = test_array[0:, 4:].astype(np.float32)

# 2層ニューラルネットワークをつくる
network = TwoLayerNet(input_size=4, hidden_size=10, output_size=3)

iters_num = 1000 # 繰り返しの回数
train_size = x_train.shape[0]
batch_size = 5 # データを5個ずつ読む
learning_rate = 0.1 # 学習率

train_loss_list = []
train_acc_list = []
test_acc_list = []

iter_per_epoch = max(train_size / batch_size, 1)

# 繰り返しの回数分だけ，学習を行う
for i in range(iters_num):
    batch_mask = np.random.choice(train_size, batch_size)
    x_batch = x_train[batch_mask]
    t_batch = t_train[batch_mask]
    
    # 勾配の計算：逆誤差伝播法で行う．numerical_gradientは使わない．
    grad = "ここを自分で書く" 
    
    # 結合荷重の更新
    for key in ('W1', 'b1', 'W2', 'b2'):
        network.params[key] -= "ここを自分で書く"
    
    # 損失関数で交差エントロピー誤差を求める
    loss = "ここを自分で書く"
    train_loss_list.append(loss)
    
    # データ1週ごとに認識精度を計算する
    if i % iter_per_epoch == 0:
        train_acc = network.accuracy(x_train, t_train)
        test_acc = network.accuracy(x_test, t_test)
        train_acc_list.append(train_acc)
        test_acc_list.append(test_acc)
        print("train acc, test acc | " + str(train_acc) + ", " + str(test_acc)) # 精度の表示
        
# グラフの描画
markers = {'train': 'o', 'test': 's'}
x = np.arange(len(train_acc_list))
plt.plot(x, train_acc_list, label='train acc')
plt.plot(x, test_acc_list, label='test acc', linestyle='--')
plt.xlabel("epochs")
plt.ylabel("accuracy")
plt.ylim(0, 1.0)
plt.legend(loc='lower right')
# plt.savefig('bp-iris.png')
plt.show()

ここまでで，逆誤差伝播法を用いた結合強度を学習するニューラルネットワークを実装できました．

逆誤差伝播法の数式はややこしいように感じたかもしれませんが，計算グラフで順伝播，逆伝播の考え方を理解すると，意外とやっていることはシンプルなことがわかります．

授業での実装はここまでとしますが，実際に研究で使うときや，製品として世の中に出すときには，結合荷重の初期値をどう決めるか，学習率をいくつに設定するか，など，パラメータチューニングについても考えなければいけません．

パラメータチューニングについては，興味があれば調べてみてください．

それでは，ここまで作業した内容を保存して，moodleから提出してください．